# Phase 3 — consolidation: final comparison tables and figures

Run **after** both `finetune_hindi_colab.ipynb` and
`finetune_nepali_kaggle.ipynb` have completed and their `*/eval/*.json` results
are in the repository. This notebook needs no GPU: it only reads results.

It produces the three artefacts the brief's Phase 3 deliverables ask for that
are not per-language — the pretrained-vs-finetuned table, the Model H versus
Model L comparison, and the error analysis — plus the accuracy figure the
report embeds.

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase3_final'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}
!pip install -q pyyaml matplotlib

In [ ]:
# Pull both languages' Phase 3 results back in.
!mkdir -p hindi/eval nepali/eval report/figures
!cp -r {DRIVE_BASE}/hindi/phase3/eval/* hindi/eval/ 2>/dev/null || true
!cp -r {DRIVE_BASE}/nepali/phase3/eval/* nepali/eval/ 2>/dev/null || true
!cp -r {DRIVE_BASE}/hindi/phase3/figures/* report/figures/ 2>/dev/null || true
!cp -r {DRIVE_BASE}/nepali/phase3/figures/* report/figures/ 2>/dev/null || true
!ls hindi/eval nepali/eval

## Comparison tables and error analysis

`compare_reasoning` reads each result's own `setup` block rather than taking
labels on the command line, so a run cannot be mislabelled in the table.

In [ ]:
!python -u -m src.eval.compare_reasoning \
    --inputs hindi/eval/reasoning_*.json nepali/eval/reasoning_*.json \
    --out report/figures/reasoning_comparison.json \
    --md  report/figures/reasoning_comparison.md \
    --figure report/figures/reasoning_accuracy.png

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('report/figures/reasoning_comparison.md').read()))

In [ ]:
from IPython.display import Image
display(Image('report/figures/reasoning_accuracy.png'))

## Attention summary, both models

The per-head detail lives in the JSON; this is the layer-level view that the
report discusses — whether finetuning moved attention closer in or further out,
and how many heads changed role.

In [ ]:
import json
for lang in ('hindi', 'nepali'):
    p = f'{lang}/eval/attention_reasoning.json'
    if not os.path.exists(p):
        print(f'{lang}: not found'); continue
    d = json.load(open(p))
    c = d['comparison']
    print(f"\n=== {lang} ===  {c['roles_changed']}/{c['heads_total']} heads changed role")
    print('layer  distance pre -> ft   entropy_norm pre -> ft   prev-token pre -> ft')
    for lay in c['per_layer']:
        dd, ee, pp = lay['mean_distance'], lay['entropy_normalised'], lay['prev_token_mass']
        print(f"{lay['layer']:>5}  {dd['pretrained']:>8.3f} -> {dd['finetuned']:<8.3f}"
              f"  {ee['pretrained']:>8.3f} -> {ee['finetuned']:<8.3f}"
              f"  {pp['pretrained']:>8.3f} -> {pp['finetuned']:<8.3f}")
    print('role transitions:', json.dumps(c['role_transitions'], indent=1))

In [ ]:
!cp report/figures/reasoning_comparison.md report/figures/reasoning_comparison.json \
   report/figures/reasoning_accuracy.png {DRIVE_BASE}/
print('Commit: report/figures/reasoning_comparison.{md,json}, '
      'report/figures/reasoning_accuracy.png')